In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import trajectory_data
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision_dataset_prepare.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dino_model import DINOFeaturePresence, DINOFeaturePresenceConcat, DINOFeaturePresenceAttnGated
from nocode_robot_programming.state_decision.dino_with_mil import DINOWithMIL
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.utils import Filename
from gesture_detector.utils import pretty_confusion_matrix
import torch
import numpy as np

from trajectory_data.skill_visualizer import play_video
from nocode_robot_programming.state_decision.utils import add_tag
from nocode_robot_programming.state_decision.plots.sparkline_plot import likelihood_sparklines_withtruelabels, likelihood_sparklines, likelihood_single_axis
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies
from nocode_robot_programming.state_decision_dataset_prepare.dataloader import ImageDatasetView, saved_img_processing
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda

seed = 48
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

from nocode_robot_programming.state_decision_dataset_prepare.dataset_D1 import d1_peg_pick, d2_peg_pick, d1_probe, d2_probe, dupl, get_dataset_view
# from nocode_robot_programming.state_decision_dataset_prepare.dataset_D2 import d1_move
datafileloader = TrajectoryDataset(trajectory_data.package_path)

You can see dataset includes these tasks, for each tasks there are several trials.

# List through the images `X` from the dataset and its labels `y`

- Tasks as load functions: `datasets = d1_peg_pick, d2_peg_pick, d1_probe, d2_probe`
- Each dataset has difficulty mixes: `hard_mix, medium_mix, easy_mix = datasets[i]`
- Each mix has train, test, and description `d_train, d_test, d_text = <easy|medium|difficult>_mix`
- `d_train` and `d_test` are 

In [ ]:
datasets = d1_peg_pick(datafileloader)
d_train, d_test, d_text = datasets[0] # hard mix
print(f"{d_text=},\n{d_train.X.shape=},\n{d_train.y_int.shape=},\n{len(d_train.y_names)=},\n{d_train.y_cls=}")

In [ ]:
# Loading artificial dataset
d_train, d_test, d_text = d1_move(datafileloader)[0]
print(f"{d_text=},\n{d_train.X.shape=},\n{d_train.y_int.shape=},\n{len(d_train.y_names)=},\n{d_train.y_cls=}")

In [ ]:
d_train.showcase_captions(fps=20, scale=2.0)
d_test.showcase_captions(fps=20, scale=2.0)

In [ ]:
d_train.play_video(fps=5, scale=2.0)
d_test.play_video(fps=10, scale=2.0)

# Checkpoint 2025-10-29

- Test accuracy achieved 93%

In [ ]:
train_stats, test_stats, model_names = [], [], []
for decider in [
        DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=None), # best model
        DINOFeaturePresenceConcat(dino_variant= "dinov2_vits14", percentile_keep=None),
        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=None, attn_mode = "hard", head_reduce = "mean", attn_keep = 0.4),
        # DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=None, attn_mode = "hard", head_reduce = "max" , attn_keep = 0.4),
        DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=None, att_hidden = 128, dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 1000),
        DINOFeaturePresence(percentile_keep=0.0, dino_variant="dinov2_vitg14"),
        # DINOFeaturePresence(dino_variant = "dinov2_vitl14", percent_keep=0),
        StateDeciderSIFT(method = "SIFT"), # suggesting the background problem
        # StateDeciderSIFT(method = "AKAZE"),
        StateDeciderSIFT(method = "ORB"),
        # AEGP(binary=False, pix=64),
        # AEGP(binary=True, pix=64),
        # AEGP(binary=False, pix=64, crop=False),
        # AEGP(binary=True, pix=64, crop=False),
    ]:
    train_stats_model, test_stats_model = [], []
    dataset_names = []
    for task_dataset_gen in [d1_peg_pick, d2_peg_pick, d1_probe, d2_probe]: #d1_move]:#, d1_peg_pick, d2_peg_pick, d1_probe, d2_probe]:
        
        hard_mid_easy = task_dataset_gen(datafileloader)

        for d_train, d_test, d_text in hard_mid_easy:
            print(f"Model: {decider.__class__.__name__}, Dataset: {d_text}")
            
            decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function

            y_pred = decider.predict_many(d_train.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False); ipt.save()
            train_stats_model.append((np.array(d_train.y_names) == np.array(y_pred)).mean())

            y_pred = decider.predict_many(d_test.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
            test_stats_model.append((np.array(d_test.y_names) == np.array(y_pred)).mean())
            
            # ipt.show(small=True)
            ipt.delete() # don't plot
            dataset_names.append(d_text.split(",")[0])

    train_stats.append(train_stats_model)
    test_stats.append(test_stats_model)
    model_names.append(decider.__str__())

In [ ]:
visualize_accuracies(100 * np.array(train_stats), 100 * np.array(test_stats), model_names, dataset_names, out_dir="plot", jupyter_plot=False)
f"Overall accuracy on train data: {round(100 * np.array(train_stats).mean())}% and test data {round(100 * np.array(test_stats).mean())}%"


# Summary:
12 Dataset variants
- 2 tasks: peg pick, probe
- 3 mixes: easy - a lot of training data, medium - 50% train and 50% test data, hard - only 2 full demonstrations of train data, each for new class, rest is test data

10 Model variants
- DINOFeaturePresence - small DINO model, 224 x 224 pixels
- DINOFeaturePresence - large DINO model, 224 x 224 pixels
    - Train: 1. Use frozen model to get features `x_norm_patchtokens`, 2. pool patches: L2-normalized mean, 3. build prototypes for each class (single-class prototype has dimension = 384 - DINO features)
    - Test: Get features `x norm patchtokens`, 2. pool patches L2-norm mean, 3. cosine similarity with the prototypes
- DINOFeaturePresence with MIL - small DINO model, 224 x 224 pixels
    - nhidden = 128, dropout = 0.1, lr = 7e-5, weight decay = 1e-3, epochs = 1000, CrossEntropyLoss, Adam optimizer, single attention head
- DINO classifier designed by ChatGPT, 224 x 224 pixels
- SIFT
- ORB
- AE-GP Multiclass (new)
- AE-GP Binary (ILeSiA)
- AE-GP Multiclass (new) - no zoom
- AE-GP Binary (ILeSiA) - no zoom
    - Videoembedder from ILeSiA
    - Input image 64 x 64 pixels
    - dataset x20 for training Video embedding
    - GP Epochs = 400
    - Latent space size = 12
    - Learning rate = 0.01

# Conclusion

The most basic DINOFeaturePresence model (`vanilla`) performed best. AE-GP, there is probably an error somewhere.

# Next steps:

- Check instance retrieval dinov2.metademolab.com
- use concatenate instead of mean
- do histogram and average the histogram

#### Play all frames that are wrongly predicted

- you can modify speed with `fps`
- you can modify window size with `scale`

In [ ]:
play_video(
    d_test.X[np.array(d_test.y_names) != np.array(decider.predict_many(d_test.X))].cpu().numpy() * 256,
    fps=2, window_name="win", backend="cv2", scale=10.0)

In [ ]:
play_video(
    d_train.X[np.array(d_train.y_names) != np.array(decider.predict_many(d_train.X))].cpu().numpy() * 256,
    fps=2, window_name="win", backend="cv2", scale=10.0)

# TODO:

- Good training depends on good starting point, it doesn't have to converge

### See if accuracy varies further from the branch timestep

In [ ]:
plt.hist(d_test.Xt[np.array(d_test.y_names) == np.array(decider.predict_many(d_test.X))].cpu().numpy())

# I want to see the likelihoods

In [ ]:
decider = DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=None, input_size=224) # best model

task_dataset_gen = d1_peg_pick #, d2_peg_pick, d1_probe, d2_probe]
hard, mid, easy = task_dataset_gen(datafileloader)
d_train, d_test, d_text = easy #, mid, hard
print(f"Model: {decider.__class__.__name__}, Dataset: {d_text}")

decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function

y_pred = decider.predict_many(d_train.X)
pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False); ipt.save()
train_stats_model = (np.array(d_train.y_names) == np.array(y_pred)).mean()

y_pred = decider.predict_many(d_test.X)
pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
test_stats_model = (np.array(d_test.y_names) == np.array(y_pred)).mean()

# ipt.show(small=True)
ipt.delete() # don't plot
dataset_name = d_text.split(",")[0]
dataset_name, train_stats_model, test_stats_model, d_train.y_cls

In [ ]:
def plots_on_view(self,view, event_x):
    logits_list = []
    for sample in view.X:
        p = self._single_patch_feats(sample)  # [M, D]
        g = self._pool_patches(p.unsqueeze(0))[0]  # [D]
        logits = self._score_logits(g.unsqueeze(0), self.prototypes)[0]  # [C]
        logits_list.append(logits.cpu().numpy()) # c = int(torch.argmax(logits).item())
    logits_list = np.array(logits_list).T
    y_int = view.y_int.cpu().numpy()

    # fig, _, _ = likelihood_sparklines_withtruelabels(
    #     likelihoods=logits_list,
    #     class_names=["original", "recovery"], #d_test.y_cls,
    #     true_labels=y_int,
    #     truth_style="barcode",      # try: 'barcode', 'strip', 'overlay', 'both'
    #     cols=2, sharey=True,
    #     )

    likelihood_single_axis(
        logits_list, ["original", "recovery"], true_labels=y_int, mode="barcode",
        figsize=(2.5, 2.0), dpi=220, event_x=event_x, event_units="index"
    )


In [ ]:
window = 20
plots_on_view(decider,
    get_dataset_view(datafileloader, tags= ['d1_peg_pick', 'd1_peg_pick_branch_at_49'], # label: 0, 1  
    at=slice(49-window/2.0, 49+window/2.0),
    file_names=['d1_peg_pick']), event_x=10); ipt.save()
plots_on_view(decider,
    get_dataset_view(datafileloader, tags= ['d1_peg_pick', 'd1_peg_pick_branch_at_49'], # label: 0, 1  
    at=slice(49-window/2.0, 49+window/2.0),
    file_names=['d1_peg_pick_branch_at_49']), 
    event_x=0); ipt.save()
ipt.show()

In [ ]:
list(zip(datafileloader.tasks["d3_probe"]["names"], datafileloader.tasks["d3_probe"]["tags"]))